In [1]:
import os

import pandas as pd
import nibabel as nb
import numpy as np

from nilearn.image import concat_imgs
from nilearn.masking import apply_mask


root = "./lss_maps"
tasks = ['gonogo', 'hcp', 'twostep', 'mid', 'risksensitive'] #posner

mask_p = "/mnt/projects/rewardMap/STUDIES/pilotstudy/derivatives/masks/tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz"

data_dict = {task : {} for task in tasks}


## load and mask data
for task in tasks:
    p = os.path.join(root, task)
    subject_dirs = [d for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))]

    for sub in subject_dirs:
        sub_dir_p = os.path.join(p, sub)

        img_files = []
        rewards_file = None

        for roor, dirs, files in os.walk(sub_dir_p):
            for file in files:
                if file.endswith(".nii.gz"):
                    img_files.append(file)
                elif file.startswith("rewards"):
                    rewards_file = os.path.join(sub_dir_p, file)

        imgs = [nb.load(os.path.join(sub_dir_p, f)) for f in sorted(img_files)]
        imgs = concat_imgs(imgs)
        masked_data = apply_mask(imgs=imgs, mask_img=mask_p)

        if rewards_file != None: #just bc the lss-map creation is still running
            rewards = pd.read_csv(rewards_file)
            rewards = rewards["reward"].to_numpy()

        data_dict[task][sub] = {"betas": masked_data, "rewards": rewards}

#data_dict

In [2]:
# # prepare data for sklearn case: leave one subject out for testing
# # shape = trials (time) x voxels

# # concatenate as extra trials

# d = data_dict["risksensitive"]

# X_train = X_test = None
# Y_train = Y_test = None

# for i, sub in enumerate(d.keys()):
#     if i == 0:
#         X_train = d[sub]["betas"]
#         Y_train = d[sub]["rewards"]
#     elif i >= len(d.keys()) - 1:
#         # last iteration
#         X_test = d[sub]["betas"]
#         Y_test = d[sub]["rewards"]

#     X_train = np.concatenate((X_train, d[sub]["betas"]), axis=0)
#     Y_train = np.concatenate((Y_train, d[sub]["rewards"]), axis=0)


# print(X_train.shape)

# classes, count = np.unique_counts(Y_train)

# print(f"baseline accuracies of: {classes} - {np.round(count/np.sum(count), 2)}")


In [3]:
# prepare data for sklearn case: leave one task out for testing
# shape = trials (time) x voxels
X_train_list = []
Y_train_list = []
X_test_list = []
Y_test_list = []

tasks = list(data_dict.keys())
test_task_idx = len(tasks) - 1  # leave last task out

for i, task in enumerate(tasks):
    d = data_dict[task]
    

    for sub in d.keys():
        betas = d[sub]["betas"]
        rewards = d[sub]["rewards"]

        if i == test_task_idx:
            X_test_list.append(betas)
            Y_test_list.append(rewards)
        else:
            X_train_list.append(betas)
            Y_train_list.append(rewards)

# concatenate along trials axis only once
X_train = np.concatenate(X_train_list, axis=0)
Y_train = np.concatenate(Y_train_list, axis=0)
X_test = np.concatenate(X_test_list, axis=0)
Y_test = np.concatenate(Y_test_list, axis=0)


In [4]:
np.save("X_train.npy", X_train)
np.save("Y_train.npy", Y_train)
np.save("X_test.npy", X_test)
np.save("Y_test.npy", Y_test)

In [1]:
import numpy as np

X_train = np.load("X_train.npy", mmap_mode='r')
Y_train = np.load("Y_train.npy")

X_test = np.load("X_test.npy", mmap_mode='r')
Y_test = np.load("Y_test.npy")

In [2]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

Y_train = np.sign(Y_train)
Y_test = np.sign(Y_test)
classes, count = np.unique_counts(Y_train)

print(f"baseline accuracies of train: {classes} - {np.round(count/np.sum(count), 2)}")

classes, count = np.unique_counts(Y_test)

print(f"baseline accuracies of test: {classes} - {np.round(count/np.sum(count), 2)}")

(7805, 235840)
(2645, 235840)
(7805,)
(2645,)
baseline accuracies of train: [-1.  0.  1.] - [0.15 0.45 0.4 ]
baseline accuracies of test: [0. 1.] - [0.29 0.71]


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
# from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", LogisticRegression(solver='saga', penalty='l1', max_iter=10000, random_state=0))
])

param_distributions = {
    'pca__n_components': [10, 50, 100],     # number of PCA components
    'model__penalty': ['l1', 'l2']       # handle imbalance
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_distributions,
    n_iter=20,                # how many random combinations to test
    cv=4,                     # 4-fold cross-validation
    scoring='balanced_accuracy',  # better metric for imbalanced data
    random_state=42,
    n_jobs=1,                # use all cores
    verbose=2
)

search.fit(X_train, Y_train)

print("Best parameters:", search.best_params_)
print("Best balanced accuracy:", search.best_score_)

# # K-Fold cross-validation
# splits = 4
# kf = KFold(n_splits=splits, shuffle=True, random_state=42)

# scores = []

# for i, (train_index, test_index) in enumerate(kf.split(X_train)):
#     X_tr = X_train[train_index]
#     X_val = X_train[test_index]
#     y_tr = Y_train[train_index]
#     y_val = Y_train[test_index]

#     pipe.fit(X_tr, y_tr)
#     score = pipe.score(X_val, y_val)
#     scores.append(score)
#     print(f"Fold {i+1}/{splits} - Validation Accuracy: {score:.4f}")

# print(f"\nMean cross-validation accuracy: {np.mean(scores):.4f}")

# # Fit on all training data and transform
# pipe.fit(X_train, Y_train)
# print("X_train shape before PCA:", X_train.shape)
# X_after = pipe.named_steps["pca"].transform(scaler.fit_transform(X_train))
# print("X_train shape after PCA:", X_after.shape)



/mnt/scratch/projects/rewardMap/reward_signature/.venv/lib64/python3.9/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=20. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 4 folds for each of 6 candidates, totalling 24 fits


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
# from sklearn.linear_model import LinearRegression

from sklearn.metrics import balanced_accuracy_score
from tqdm import tqdm

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=25)),
    ("model", LogisticRegression(solver='saga', penalty=None, max_iter=10000, random_state=0))
])

# K-Fold cross-validation
splits = 4
kf = KFold(n_splits=splits, shuffle=True, random_state=42)

scores = []

for i, (train_index, test_index) in tqdm(enumerate(kf.split(X_train))):
    X_tr = X_train[train_index]
    X_val = X_train[test_index]
    y_tr = Y_train[train_index]
    y_val = Y_train[test_index]

    pipe.fit(X_tr, y_tr)


    y_pred = pipe.predict(X_tr)
    score = balanced_accuracy_score(y_tr, y_pred)
    scores.append(score)
    
    print(f"Fold {i+1}/{splits} - Validation Accuracy: {score:.4f}")

print(f"\nMean cross-validation accuracy: {np.mean(scores):.4f}")


0it [05:41, ?it/s]


ValueError: Found input variables with inconsistent numbers of samples: [5853, 7805]